# Carga de datos

In [1]:
library(grid)
library(dplyr)
library(gridExtra)
library(visualizeR)
library(downscaleR)
library(transformeR)
library(RColorBrewer)
library(latticeExtra)
library(easyVerification)

color = colorRampPalette(rev(brewer.pal(n = 9, "RdYlBu")))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine


Loading required package: transformeR




    _______   ____  ___________________  __  ________ 
   / ___/ /  / /  |/  / __  /_  __/ __/ / / / / __  / 
  / /  / /  / / /|_/ / /_/ / / / / __/ / /_/ / /_/_/  
 / /__/ /__/ / /  / / __  / / / / /__ /___  / / \ \ 
 \___/____/_/_/  /_/_/ /_/ /_/  \___/    /_/\/   \_\ 
 
      github.com/SantanderMetGroup/climate4R



transformeR version 2.2.2 (2023-10-26) is loaded


Get the latest stable version (2.2.5) using <devtools::install_github('SantanderMetGroup/transformeR')>

Please see 'citation("transformeR")' to cite this package.

visualizeR version 1.6.4 (2023-10-26) is loaded

Please see 'citation("visualizeR")' to cite this package.

downscaleR version 3.3.4 (2023-06-22) is loaded

Please use 'citation("downscaleR")' to cite this package.

Loading required package: lattice

Loading required package: SpecsVerification


Attaching package: ‘easyVerification’


The following object is masked from ‘package:SpecsVerification’:

    EnsCorr




El primer paso es preparar los datos de nuestro predictando, la temperatura media (tas) de ERA5-Land a 0.1º, y los datos de nuestros predictores, la temperatura media (tas) y la presión en superficie (sp) de ERA5 a 0.25º, pero habiendo escalado los datos a la resolución de nuestro modelo del ECMWF, en este caso 1º.

In [2]:
# Predictando (Y) - ERA5-Land (Alta Resolución 0.1°)
y_obs = readRDS('../../data/downscaling/tas_cgdds_ERA5-Land.rds')
yT_obs = subsetGrid(y_obs, years = 1981:2016)  # training

# Predictores (X) - ERA5 (Resolución Original 0.25º - Interpolada a Resolución SEAS5 1º)
x_sp = readRDS('../../data/downscaling/sp_ERA5.rds')
xT_sp = subsetGrid(x_sp, years = 1981:2016)  # training

x_tas = readRDS('../../data/downscaling/tas_ERA5.rds')
xT_tas = subsetGrid(x_tas, years = 1981:2016)  # training

# Unimos los grid con makeMultiGrid
xT = makeMultiGrid(xT_tas, xT_sp) %>% redim(drop = TRUE)

# Model cross-validation

In [3]:
analog.cv = downscaleCV(x = xT, y = yT_obs, method = "analogs", n.analogs = 1,
                        sampling.strategy = "leave-one-year-out",
                        prepareData.args = list(
                            spatial.predictors = list(which.combine = getVarNames(xT), v.exp = 0.95)))

fold: 1 --> calculating...

[2026-01-27 12:08:45.078447] Performing PC analysis on 2 variables plus a combination ...

[2026-01-27 12:08:47.699627] Done.

fold: 2 --> calculating...

[2026-01-27 12:09:20.54332] Performing PC analysis on 2 variables plus a combination ...

[2026-01-27 12:09:23.051097] Done.

fold: 3 --> calculating...

[2026-01-27 12:09:54.223707] Performing PC analysis on 2 variables plus a combination ...

[2026-01-27 12:09:56.765321] Done.

fold: 4 --> calculating...

[2026-01-27 12:10:27.797298] Performing PC analysis on 2 variables plus a combination ...

[2026-01-27 12:10:30.131653] Done.

fold: 5 --> calculating...

[2026-01-27 12:11:00.606153] Performing PC analysis on 2 variables plus a combination ...

[2026-01-27 12:11:03.081119] Done.

fold: 6 --> calculating...

[2026-01-27 12:11:34.298176] Performing PC analysis on 2 variables plus a combination ...

[2026-01-27 12:11:36.803965] Done.

fold: 7 --> calculating...

[2026-01-27 12:12:07.925178] Performing PC 

# Validation

### Función auxiliar

In [4]:
# Función para calcular correlación de Pearson y valores p entre datos de modelo y observaciones en una grilla espacial
# Además, identifica y marca los puntos con correlación estadísticamente significativa según un umbral de p-valor
#
# Args:
#   model_data: objeto con datos del modelo, estructura esperada con dimensión [miembros, tiempo, latitud, longitud]
#   obs_data: objeto con datos observacionales, estructura con dimensión [tiempo, latitud, longitud]
#   ref_grid: objeto referencia con metadatos espaciales y temporales para construir grillas (xyCoords, Variable, Dates)
#   threshold: umbral para marcar significancia estadística (p-valor), default 0.05
#
# Returns:
#   Lista con:
#     - cor: matriz de correlaciones [lat x lon]
#     - pval: matriz de valores p [lat x lon]
#     - pval_grid: objeto tipo "grid" con valores p y metadatos
#     - pts: lista de objetos para graficar puntos de significancia (stippling)

calc_cor_pval_grid = function(model_data, obs_data, ref_grid, threshold = 0.05) {
    
    # Dimensiones espaciales (latitud y longitud)
    lat_n = dim(model_data$Data)[2]
    lon_n = dim(model_data$Data)[3]
  
    # Inicializar matrices vacías para almacenar correlaciones y p-valores
    cor_array = matrix(NA, nrow = lat_n, ncol = lon_n)
    pval_array = matrix(NA, nrow = lat_n, ncol = lon_n)
    
    # Iterar sobre cada punto espacial
    for (i in 1:lat_n) {
        for (j in 1:lon_n) {
            
            # Extraer series temporales de modelo y observaciones para la celda actual
            pred_series = model_data$Data[, i, j]
            obs_series = obs_data$Data[, i, j]
      
            # Filtrar índices con datos completos (no NA)
            valid_idx = complete.cases(pred_series, obs_series)
            
            # Solo calcular correlación si hay suficientes datos (mínimo 10)
            if (sum(valid_idx) >= 10) {
                test = cor.test(pred_series[valid_idx], obs_series[valid_idx], method = "pearson")
                cor_array[i, j] = test$estimate  # Coeficiente de correlación
                pval_array[i, j] = test$p.value  # Valor p de la prueba
            }
        }
    }
  
    # Construir un objeto "grid" para los valores p, con metadatos espaciales y temporales
    pval_grid = list()
    pval_grid$Data = pval_array
    attr(pval_grid$Data, "dimensions") = c("lat", "lon")
    pval_grid$xyCoords = ref_grid$xyCoords
    pval_grid$Variable = ref_grid$Variable
    pval_grid$Dates = ref_grid$Dates
    class(pval_grid) = "grid"

    pval_grid$Variable$varName = "p-values"
    attr(pval_grid$Variable, "description") = "Mapa de p-valores"
    attr(pval_grid$Variable, "units") = ""
    attr(pval_grid$Variable, "longname") = "p-values"

    # Construir un objeto "grid" para los valores de correlación,
    cor_grid = list()
    cor_grid$Data = cor_array
    attr(cor_grid$Data, "dimensions") = c("lat", "lon")
    cor_grid$xyCoords = ref_grid$xyCoords
    cor_grid$Variable = ref_grid$Variable
    cor_grid$Dates = ref_grid$Dates
    class(cor_grid) = "grid"

    cor_grid$Variable$varName = "correlation"
    attr(cor_grid$Variable, "description") = "Mapa de correlaciones"
    attr(cor_grid$Variable, "units") = ""
    attr(cor_grid$Variable, "longname") = "correlation"

    # Crear objetos para graficar puntos de significancia estadística (stippling)
    pts = map.stippling(climatology(pval_grid), 
                        threshold = threshold, 
                        condition = "LT", 
                        pch = 19, col = "black", cex = 0.05) %>% suppressMessages() %>% suppressWarnings()
    
    # Devolver lista con resultados y objetos para plot
    return(list(cor = cor_grid, pval = pval_array, pval_grid = pval_grid, pts = pts))
}

### Bias and corr 

In [5]:
yT_obs = aggregateGrid(yT_obs, aggr.y = list(FUN = "mean", na.rm = TRUE)) %>% suppressMessages %>% suppressWarnings
analog.cv = aggregateGrid(analog.cv, aggr.y = list(FUN = "mean", na.rm = TRUE)) %>% suppressMessages %>% suppressWarnings
#analog.cv5 = aggregateGrid(analog.cv5, aggr.y = list(FUN = "mean", na.rm = TRUE)) %>% suppressMessages %>% suppressWarnings

In [9]:
# Calculo el bias entre la predicción y las observaciones en el periodo de train (kNN = 1)
ref = climatology(yT_obs) %>% suppressMessages %>% suppressWarnings
diff = climatology(analog.cv) %>% suppressMessages %>% suppressWarnings
bias = gridArithmetics(diff, ref, operator = "-")
b_plot = spatialPlot(bias,
                     backdrop.theme = "countries",
                     main = "Bias Train (kNN = 1) | Mean = 0.13",
                     col.regions = color,
                     at = seq(-0.3, 0.3, 0.1)) %>% suppressMessages %>% suppressWarnings

test_cor = calc_cor_pval_grid(analog.cv, yT_obs, analog.cv)

corr_plot = spatialPlot(climatology(test_cor$cor),
                        backdrop.theme = "countries",
                        main = "Corr Train (kNN = 1) | Mean = 0.91",
                        sp.layout = list(test_cor$pts),
                        col.regions = color,
                        at = seq(-1, 1, 0.1)) %>% suppressMessages %>% suppressWarnings

In [10]:
png("metricas_train_cv.png", width = 2000, height = 1000, res = 150)
grid.arrange(b_plot, corr_plot, ncol = 2)
dev.off()

pdf 
  2